# Data Mining: Homework assignment (part 2)

### Theofanis Barmparosos: sdi2200107

### Georgios Spyrou: sdi2200168

Dependencies

In [1]:
%pip install seaborn
%pip install datasets
%pip install matplotlib
%pip install vaderSentiment
%pip install kneed
%pip install contractions
%pip install pyspellchecker

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated pa

For the csv files that this notebook creates, you can edit the number of max_rows below, to load as many rows from the datasets that you desire. We have tested the code on 1000, 10000 and 100000 lines, and the results are of course much better with a higher number of rows.  
We must however warn, that when max_rows exceeds values around 20000, it will require a lot of memory space(RAM) and is very likely to crash the jupyter kernel at the execution of some cell.

In [2]:
# limiting the csv to max_rows to work first
max_rows = 100000000

In [3]:
#########################################
###                                   ###
###           DATA FETCHING           ###
###                                   ###
#########################################
from datasets import load_dataset

print("Reviews:")

beauty_reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_All_Beauty", split="full", streaming=True, trust_remote_code=True)
for row in beauty_reviews:
    print(row)
    break #print only the first row for testing.

appliances_reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Appliances", split="full", streaming=True, trust_remote_code=True)
for row in appliances_reviews:
    print(row)
    break #print only the first row for testing.

musical_instruments_reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Musical_Instruments", split="full", streaming=True, trust_remote_code=True)
for row in musical_instruments_reviews:
    print(row)
    break #print only the first row for testing.

software_reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Software", split="full", streaming=True, trust_remote_code=True)
for row in software_reviews:
    print(row)
    break #print only the first row for testing.

video_games_reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Video_Games", split="full", streaming=True, trust_remote_code=True)
for row in video_games_reviews:
    print(row)
    break #print only the first row for testing.


print("Meta data:")

beauty_meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_All_Beauty", split="full", streaming=True, trust_remote_code=True)
for row in beauty_meta:
    print(row)
    break #print only the first row for testing.

appliances_meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Appliances", split="full", streaming=True, trust_remote_code=True)
for row in appliances_meta:
    print(row)
    break #print only the first row for testing.

musical_instruments_meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Musical_Instruments", split="full", streaming=True, trust_remote_code=True)
for row in musical_instruments_meta:
    print(row)
    break #print only the first row for testing.

software_meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Software", split="full", streaming=True, trust_remote_code=True)
for row in software_meta:
    print(row)
    break #print only the first row for testing.

video_games_meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Video_Games", split="full", streaming=True, trust_remote_code=True)
for row in video_games_meta:
    print(row)
    break #print only the first row for testing.


# our categories in a list for easy access
categories = ["appliances", "beauty", "musical_instruments", "software", "video_games"]
# debugging
# categories = ["appliances"]

Reviews:
{'rating': 5.0, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': [], 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1588687728923, 'helpful_vote': 0, 'verified_purchase': True}


KeyboardInterrupt: 

##### CSV creation

Creating the CSV files from streaming, with, only the necessary lines until max_rows.  
**! Important !**  
For this part we chose to work only with the category video_games, entirely because of the limited resources. More than one category for more than ~20000 lines, would risk the kernel crashing before it finishes executing the notebook when ran on our devices.

In [ ]:
# CSV creation

import csv
import os

# data_dict is one of the two dictionairies reviews/meta_data
# data_type is either "reviews" or "meta"
# max_rows is the number of rows to be written in the csv file
def write_csv_files(data_dict, data_type, max_rows):
    first_category = next(iter(data_dict))
    for first_row in data_dict[first_category]:
        columns = list(first_row.keys())
        break
    for category, data in data_dict.items():
        if not data:
            print(f"No data for {category} {data_type}")
            continue
        filename = f"./{category}_{data_type}.csv"
        print(f"Writing {category}_{data_type}.csv...")
        try:
            with open(filename, "w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file, escapechar='\\')
                writer.writerow(columns)
                count = 0
                for row in data:
                    if count >= max_rows:
                        break
                    csv_row = [row.get(col, "") for col in columns]
                    writer.writerow(csv_row)
                    count += 1
            print(f"Finished writing {filename} with {count} rows")
        except Exception as e:
            print(f"Error writing {filename}: {e}")


reviews = {
    "appliances": appliances_reviews,
    "beauty": beauty_reviews,
    "musical_instruments": musical_instruments_reviews,
    "software": software_reviews,
    "video_games": video_games_reviews
}

meta_data = {
    "appliances": appliances_meta,
    "beauty": beauty_meta,
    "musical_instruments": musical_instruments_meta,
    "software": software_meta,
    "video_games": video_games_meta
}

write_csv_files(reviews, "reviews", max_rows)
write_csv_files(meta_data, "meta", max_rows)





# The lines below produce the same result. They aren't optimal because,
# they fetch the whole dataset first and then only keep the max_rows whereas
# with the method above, we are able to fetch and create the csv files with
# the necessary lines only("max_rows"). And this is why it isn't preferred.


# pd.DataFrame(appliances_reviews).head(max_rows).to_csv("appliances_reviews.csv", index=False)
# pd.DataFrame(beauty_reviews).head(max_rows).to_csv("beauty_reviews.csv", index=False)
# pd.DataFrame(musical_instruments_reviews).head(max_rows).to_csv("musical_instruments_reviews.csv", index=False)
# pd.DataFrame(software_reviews).head(max_rows).to_csv("software_reviews.csv", index=False)
# pd.DataFrame(video_games_reviews).head(max_rows).to_csv("video_games_reviews.csv", index=False)

# pd.DataFrame(appliances_meta).head(max_rows).to_csv("appliances_meta.csv", index=False)
# pd.DataFrame(beauty_meta).head(max_rows).to_csv("beauty_meta.csv", index=False)
# pd.DataFrame(musical_instruments_meta).head(max_rows).to_csv("musical_instruments_meta.csv", index=False)
# pd.DataFrame(software_meta).head(max_rows).to_csv("software_meta.csv", index=False)
# pd.DataFrame(video_games_meta).head(max_rows).to_csv("video_games_meta.csv", index=False)

##### Dataframes from CSVs

In [ ]:
import pandas as pd

# We need two dictionaries, one for reviews dataframes, one for meta dataframes
reviews_dfs = {}
meta_dfs = {}
for category in categories:
    reviews_csv_file = f"./{category}_reviews.csv"
    meta_csv_file = f"./{category}_meta.csv"
    reviews_dfs[category] = pd.read_csv(reviews_csv_file)
    meta_dfs[category] = pd.read_csv(meta_csv_file)
    print(f"Created dataframes for {category}")


We're starting off with some basic preprocessing and normalization of the dataset. When doing wordclouds we noticed that a lot of HTML elements would show up, so we are removing HTML elements from the review texts. Another issue were the missing values for the price column, in the metadata datasets. We can't have "None" values for fileds as important as prices, so we decided to replace "None" price values, with the mean price of the whole dataset.

In [ ]:
###########################################
###                                     ###
###       HANDLING MISSING VALUES       ###
###                                     ###
###########################################

def display_number_of_null(df):
    columns = [
        'price', 'average_rating', 'rating_number', 'features',
        'description', 'images', 'videos', 'store', 'categories',
        'details', 'parent_asin', 'bought_together', 'subtitle', 'author'
    ]

    for col in columns:
        col_sum = df[col].isnull().sum()    
        print(f"Column \"{col}\" has {col_sum} null values")
    print("\n")

for category, df in meta_dfs.items():
    text = f" {category} "
    pad = 40 - len(text)
    left = pad // 2
    right = pad - left
    print("=" * left + text + "=" * right)
    display_number_of_null(df)


def get_mean_price_from_df(df):
    # converts column price to numbers, and non-numerics to NaN
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    return df['price'].mean()

for category in categories:
    if category == "appliances":
        appliances_mean = get_mean_price_from_df(meta_dfs[category])
        print(f'The average product price in category "{category}" is', appliances_mean)
    if category == "beauty":
        beauty_mean = get_mean_price_from_df(meta_dfs[category])
        print(f'The average product price in category "{category}" is', beauty_mean)
    if category == "musical_instruments":
        musical_instruments_mean = get_mean_price_from_df(meta_dfs[category])
        print(f'The average product price in category "{category}" is', musical_instruments_mean)
    if category == "software":
        software_mean = get_mean_price_from_df(meta_dfs[category])
        print(f'The average product price in category "{category}" is', software_mean)
    if category == "video_games":
        video_games_mean = get_mean_price_from_df(meta_dfs[category])
        print(f'The average product price in category "{category}" is', video_games_mean)

        

In [ ]:
def replace_invalid_prices(df, mean):
    df['price'] = pd.to_numeric(df['price'], errors='coerce') # coerce replaces any non-numeric value with NaN
    # locates the rows that have bad 'price' value and replaces the 'price' in these rows with the mean value provided
    df.loc[(df['price'].isna()) | (df['price'] <= 0.0), 'price'] = mean
    return df

# replacing all bad values with the mean value of the rest
for category in categories:
    if category == "appliances":
        replace_invalid_prices(meta_dfs['appliances'], appliances_mean)
    if category == "beauty":
        replace_invalid_prices(meta_dfs['beauty'], beauty_mean)
    if category == "musical_instruments":
        replace_invalid_prices(meta_dfs['musical_instruments'], musical_instruments_mean)
    if category == "software":
        replace_invalid_prices(meta_dfs['software'], software_mean)
    if category == "video_games":
        replace_invalid_prices(meta_dfs['video_games'], video_games_mean)

# these should be now zero
for category in categories:
    df_nulls = (meta_dfs[category])['price'].isnull().sum()
    print(f"{df_nulls}")

### Task 1: Clustering for Product Grouping

In [ ]:
#######################################################
###                                                 ###
###          META DATA TEXT PRE-PROCESSING          ###
###                                                 ###
#######################################################

import re
import html
import contractions
from spellchecker import SpellChecker

emoji_dict = {
    r":\)": "smile",
    r":D": "smile",
    r";\)": "wink",
    r":\(": "sad",
    r":'\(": "cry",
    r":P": "playful",
    r"XD": "laugh",
    r"xD": "laugh",
    r"<3": "love",
    r":O": "surprise"
}

def handle_simple_emojis(text):
    for pattern, replacement in emoji_dict.items():
        text = re.sub(pattern, replacement, text)
    return text

def remove_contractions(text):
    # expand contractions (e.g haven't -> have not)
    return contractions.fix(text)

# initialize spell checker
spell = SpellChecker(distance=1)

def my_spellcheck(text):
    words = text.split()
    corrected_words = []
    for word in words:
        # if it's wrong
        if word not in spell:
            test_word = spell.correction(word)
            # found correction
            if test_word is not None:
                corrected_words.append(test_word)
            # didn't find correction
            else:
                corrected_words.append(word)
        # else, if it's correct, append it to the corrected
        else:
            corrected_words.append(word)
    return " ".join(corrected_words)

def text_cleanup(text):
    text = text.lower() # lowercase everything
    text = handle_simple_emojis(text) # handling basic emojis
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'email', text) # handle email adresses
    text = re.sub(r'http[s]?://\S+', 'url', text) # handle urls
    text = re.sub(r'@\w+', 'username', text) # handle mentions
    text = re.sub(r'#(\w+)', r'\1', text) # handle hashtags: remove the symbol, keep the text
    text = html.unescape(text) # handling html tags
    text = re.sub(r'(\w)\1{2,}', r'\1\1', text) # handling unecessarily long words (e.g. soooooo -> so)
    text = remove_contractions(text) # handling contractions (haven't -> have not)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text) # remove anything that is not english, a number or a whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # we didn't use pyspellchecker because it would remove significant points from
    # the text that could be vital for text manipulation later, also it would take
    # too much time
    # text = my_spellcheck(text) # spellchecking using pyspellchecker
    return text

# apply cleanup to all datasets
print("Cleaning data...")
for category in categories:
    print(f"{'=' * 18} {category.center(21)} {'=' * 18}")
    re_df = meta_dfs[category]
    re_df['description'] = re_df['description'].fillna("").astype(str)
    re_df['description'] = re_df['description'].apply(text_cleanup)
    print(re_df['description'].head())
    print("\n")

In [ ]:
#############################################################
###                                                       ###
###           TF-IDF ENCODING OF META DATA TEXT           ###
###                                                       ###
#############################################################

from sklearn.preprocessing import MaxAbsScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
import numpy as np

# initialize TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,  # limit vocabulary size for efficiency
    stop_words='english',  # remove English stopwords
    ngram_range=(1, 3),  # consider both single words and bi-grams
    dtype=np.float32
)

# dictionary to store TF-IDF matrices for each category
tfidf_matrices = {}

for category in categories:
    print(f"Processing TF-IDF for {category}...")
    
    # cleaned text data (description)
    texts = meta_dfs[category]['description']
    # texts = meta_dfs[category]['description'].fillna('')
    # fit and transform
    tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
    tfidf_matrices[category] = tfidf_matrix
    
    print(f"Shape of TF-IDF matrix for {category}: {tfidf_matrix.shape}\n")


feature_matrices = {}
for category in categories:
    numerical_data = meta_dfs[category][['price', 'average_rating']].fillna(0).astype(np.float32)

    scaler = MaxAbsScaler()
    numerical_scaled = scaler.fit_transform(numerical_data)
    
    feature_matrices[category] = hstack([tfidf_matrices[category], csr_matrix(numerical_scaled)])
    print(f"TF-IDF shape for {category}: {tfidf_matrices[category].shape}, "
      f"Numerical shape: {numerical_scaled.shape}, "
      f"Combined shape: {feature_matrices[category].shape}")

In [ ]:
##############################################
###                                        ###
###               CLUSTERING               ###
###                                        ###
##############################################

from sklearn.cluster import MiniBatchKMeans
# from sklearn.cluster import KMeans
from kneed import KneeLocator
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD
# from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import gc

# Finding the optimal number of clusters for each category first

# we will use the elbow method to determine which number of
# clusters provides the optimal insertia value
# (sum of squares of distances from the center of the cluster)
K_range = list(range(4, 21))
optimal_k = {}
silhouette_scores = {}
for category in categories:
    print(f"Running Elbow Method for category: {category}")
    X = feature_matrices[category]
    inertia = []

    for k in K_range:
        kmeans = MiniBatchKMeans(
            n_clusters=k, 
            random_state=42, 
            batch_size=8192,
            n_init=3
        )
        kmeans.fit(X)
        inertia.append(kmeans.inertia_)

    kneedle = KneeLocator(K_range, inertia, curve='convex', direction='decreasing')
    optimal_k[category] = kneedle.elbow
    if optimal_k[category] is None:
        optimal_k[category] = 6
        print("Warning: No clear elbow found. Defaulting to k=6.")
    else:
        print(f"Optimal number of clusters for {category}: {optimal_k[category]}")

    # plot for each category
    plt.figure(figsize=(12, 8))
    plt.xticks(K_range)
    plt.plot(K_range, inertia, marker='o')
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Inertia')
    plt.title(f'Elbow Plot - {category}')
    plt.grid(True)
    plt.show()

    print(f"Performing (mini-batch)KMeans clustering for \"{category}\"...")

    final_model = MiniBatchKMeans(
        n_clusters=optimal_k[category],
        random_state=42,
        batch_size=16384,
        n_init=30,
        max_no_improvement=100,      # give time for improvement
        reassignment_ratio=0.001     # helps to not leave any cluster empty
    )
    cluster_labels = final_model.fit_predict(X)

    try:
        if X.shape[0] > 30000:
            score = silhouette_score(X, cluster_labels, sample_size=30000, random_state=42)
            print(f"Silhouette Score (sampled 30k): {score:.4f}")
        else:
            score = silhouette_score(X, cluster_labels)
            print(f"Silhouette Score (full): {score:.4f}")
        silhouette_scores[category] = score
    except Exception as e:
        print(f"Could not compute silhouette: {e}")

    print("Generating Visualization...")
    svd = TruncatedSVD(n_components=2, random_state=42)
    X_reduced = svd.fit_transform(X)

    plt.figure(figsize=(20, 10))
    scatter = plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=cluster_labels, cmap='tab10', s=10, alpha=0.3)
    plt.title(f"Clustering Visualization (SVD) - {category} (k={optimal_k[category]})")
    plt.xlabel("SVD Component 1")
    plt.ylabel("SVD Component 2")
    plt.colorbar(scatter, label='Cluster')
    plt.grid(True, alpha=0.3)
    plt.show()
    print("\n\n\n\n")

    del kmeans, final_model, cluster_labels, X_reduced, svd, inertia
    gc.collect()


### Task 2: Recommendation System

In [ ]:
#############################################################
###                                                       ###
###       FINAL OPTIMIZED RECOMMENDATION SYSTEM           ###
###       (Content-Based + User-Based + Hybrid)           ###
###                                                       ###
#############################################################

import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

# --- making User-Item matrices ---
user_item_matrices = {}
user_index_maps = {}    # { Index: UserID }
item_index_maps = {}    # { Index: ASIN }

print("\n--- Building Interaction Matrices ---")
for category in categories:
    if category not in reviews_dfs: continue
    
    df = reviews_dfs[category][['user_id', 'parent_asin', 'rating']].dropna()

    # Pandas categorical για γρήγορο mapping και λιγότερη μνήμη
    user_c = df['user_id'].astype('category')
    item_c = df['parent_asin'].astype('category')

    # store mappings (index -> real ID)
    user_index_maps[category] = dict(enumerate(user_c.cat.categories))
    item_index_maps[category] = dict(enumerate(item_c.cat.categories))

    # creating CSR(compressed sparse row) matrix
    matrix = csr_matrix(
        (df['rating'], (user_c.cat.codes, item_c.cat.codes)),
        shape=(len(user_c.cat.categories), len(item_c.cat.categories))
    )
    user_item_matrices[category] = matrix
    print(f"[{category}] Interaction Matrix shape: {matrix.shape}")



# --- metadata mappings & titles ---
meta_id_maps = {}   # { ASIN: index in feature matrix }
asin_to_title = {}  # { ASIN: title }

print("\n--- Building Metadata Maps ---")
for category in categories:
    if category not in meta_dfs: continue

    meta_df = meta_dfs[category].reset_index(drop=True)
    
    # ASIN -> feature matrix index
    meta_id_maps[category] = {asin: i for i, asin in enumerate(meta_df['parent_asin'])}
    
    # ASIN -> title
    meta_clean = meta_df[['parent_asin', 'title']].dropna(subset=['parent_asin'])
    asin_to_title[category] = dict(zip(meta_clean['parent_asin'], meta_clean['title']))
    print(f"[{category}] Mapped titles for {len(asin_to_title[category])} items")


# --- trining KNN(K-nearest neighbors) models ---
content_knn_models = {} 
user_knn_models = {}
print("\n--- Training KNN Models ---")

for category in categories:
    # content-based model
    # looks for similarity based on TF-IDF features
    if category in feature_matrices:
        print(f"Fitting Content-Based KNN for {category}...")
        cb_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=21, n_jobs=-1)
        cb_model.fit(feature_matrices[category])
        content_knn_models[category] = cb_model
    # user-based model
    # looks for similarity based on user's previous purchases
    if category in user_item_matrices:
        print(f"Fitting User-Based KNN for {category}...")
        ub_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=21, n_jobs=-1)
        ub_model.fit(user_item_matrices[category])
        user_knn_models[category] = ub_model


# recommendation functions

# helper to display titles instead of ASINs
def get_title(asin, category):
    return asin_to_title[category].get(asin, asin)

# Content-Based:
# given an item's ID, finds similar items based on description
def recommend_items_content_based(item_id, category, top_n=5):
    if category not in meta_id_maps or category not in content_knn_models: return []

    meta_map = meta_id_maps[category]
    if item_id not in meta_map: return []

    # find the item's index and ask the model for its nearest neighbors
    item_idx = meta_map[item_id]
    query_vector = feature_matrices[category][item_idx]
    distances, indices = content_knn_models[category].kneighbors(query_vector)

    # indices -> ASINs
    inv_meta_map = {v: k for k, v in meta_map.items()}
    recommendations = []

    for idx in indices.flatten()[1:]: # skip self (first result is the original product itself)
        rec_id = inv_meta_map.get(idx)
        if rec_id: recommendations.append(rec_id)
        if len(recommendations) >= top_n: break
            
    return recommendations

# User-Based:
# given an item's ID, finds items that appeal to similar users
def recommend_items_user_based(user_id, category, top_n=5):
    if category not in user_knn_models: return []
    
    # user ID -> index
    user_to_idx = {v: k for k, v in user_index_maps[category].items()}
    if user_id not in user_to_idx: return []
    
    user_idx = user_to_idx[user_id]
    
    # find the closest users
    user_vector = user_item_matrices[category][user_idx]
    distances, neighbor_indices = user_knn_models[category].kneighbors(user_vector)
    neighbor_indices = neighbor_indices.flatten()[1:] # skip self (first result is the original user himself)
    
    if len(neighbor_indices) == 0: return []

    # calculate the mean among the neighbors' ratings
    neighbor_ratings = user_item_matrices[category][neighbor_indices]
    
    predicted_ratings = np.array(neighbor_ratings.mean(axis=0)).flatten()
    
    # filter out the ones that the user has already seen/reviewed (shouldn't recommend those)
    user_seen = user_item_matrices[category][user_idx].toarray().flatten()
    predicted_ratings[user_seen > 0] = 0
    
    # keep the top indices
    top_indices = predicted_ratings.argsort()[::-1][:top_n]
    
    # index -> ASIN
    idx_to_item = item_index_maps[category]
    recommendations = [idx_to_item[idx] for idx in top_indices if predicted_ratings[idx] > 0]
    
    return recommendations

# Hybrid recommendation system:
# 1. attempts User-Based
# 2. if not enough recommendations are made (new use or sparse data)
# we complement with content-based using the last product the user rated
def recommend_items_hybrid(user_id, category, top_n=5):
    # User-Based recommendations
    ub_recs = recommend_items_user_based(user_id, category, top_n=top_n)
    # if we get enough recommendations, return
    if len(ub_recs) >= top_n:
        return ub_recs[:top_n]
    
    # otherwise we will complement with content-based recommendations
    user_to_idx = {v: k for k, v in user_index_maps[category].items()}
    if user_id in user_to_idx:
        u_idx = user_to_idx[user_id]
        user_ratings = user_item_matrices[category][u_idx].toarray().flatten()
        rated_indices = np.where(user_ratings > 0)[0]
        
        if len(rated_indices) > 0:
            last_item_idx = rated_indices[-1] 
            last_item_id = item_index_maps[category][last_item_idx]
            
            cb_recs = recommend_items_content_based(last_item_id, category, top_n=top_n)
            
            # we prioritize the user-based results when connecting the two lists
            final_recs = list(dict.fromkeys(ub_recs + cb_recs))
            return final_recs[:top_n]
            
    return ub_recs

# Evaluation Function (Recall@K)
def recall_at_k(user_id, category, recommend_func, k=5):
    # calculates recall@k score, how many of the hidden items we will find
    user_to_idx = {v: k for k, v in user_index_maps[category].items()}
    if user_id not in user_to_idx: return None
    
    u_idx = user_to_idx[user_id]
    
    # find what he has rated
    user_row = user_item_matrices[category][u_idx].toarray().flatten()
    rated_indices = np.where(user_row > 0)[0]
    
    if len(rated_indices) < 2: return None # not enough data
    
    # choose a random item to hide (Leave-One-Out)
    test_idx = random.choice(rated_indices)
    test_item_id = item_index_maps[category][test_idx]
    
    # we temporarily hide it from the matrix
    original_val = user_item_matrices[category][u_idx, test_idx]
    user_item_matrices[category][u_idx, test_idx] = 0
    
    try:
        if recommend_func == recommend_items_content_based:
            # for the content based recommendation, we provide another product of the same user as input
            other_idx = [i for i in rated_indices if i != test_idx][0]
            other_item_id = item_index_maps[category][other_idx]
            recs = recommend_func(other_item_id, category, k)
        else:
            recs = recommend_func(user_id, category, k)
    except:
        recs = []
        
    # revert the value in the matrix
    user_item_matrices[category][u_idx, test_idx] = original_val
    
    # check to see if the hidden item is in the recommendations
    return 1 if test_item_id in recs else 0







print("\n===== Testing Recommendations (with Titles) =====")
for category in categories:
    try:
        print(f"#################################################")
        print("##")
        print(f"##       Category: {category}")
        print("##")
        print(f"#################################################\n")
        k=5

        # Content-Based recommendations
        test_asin = list(meta_id_maps[category].keys())[0]
        test_item_title = asin_to_title[category].get(test_asin, test_asin)
        
        print(f"----- Content-Based recommendations for item: '{test_item_title}' -----")
        recs_cb = recommend_items_content_based(test_asin, category, k)
        for i, r in enumerate(recs_cb, 1):
            print(f"{i}. {get_title(r, category)}")
        
        # User-Based recommendations
        for idx in range(50):
            test_user = list(user_index_maps[category].values())[idx]
            print(f"\n----- User-Based recommendations for user: {test_user} -----")
            recs_ub = recommend_items_user_based(test_user, category, k)
            if not recs_ub:
                print("No recommendations found (User has too few distinct neighbors or data is sparse).")
            else:
                for i, r in enumerate(recs_ub, 1):
                    print(f"{i}. {get_title(r, category)}")
                break

        # Hybrid recommendations
        test_user = list(user_index_maps[category].values())[10]
        print(f"\n----- Hybrid recommendations for user: {test_user} -----")
        recs_hybrid = recommend_items_hybrid(test_user, category, k)
        for i, r in enumerate(recs_hybrid, 1):
            print(f"{i}. {get_title(r, category)}")
        
        # Evaluation
        score = recall_at_k(test_user, category, recommend_items_hybrid, k)
        print(f"\nRecall@{k} Score: {score}")
        
    except Exception as e:
        print(f"Error testing {category}: {e}")

In [ ]:
# evaluating results of the three methods based on the 10 first users

video_games_users = user_item_matrices["video_games"].index[:10]

results = []

for user_id in video_games_users:
    print(f"Evaluating user: {user_id}")

    rec_cf = recall_at_k(user_id, "video_games", recommend_items_user_based, k=5)
    rec_cbf = recall_at_k(user_id, "video_games", recommend_items_content_based, k=5)
    rec_hybrid = recall_at_k(
        user_id,
        "video_games",
        lambda u, c, k: recommend_items_hybrid(u, c, k, 0.7, 0.3),
        k=5
    )

    results.append({
        "user_id": user_id,
        "CF": round(rec_cf, 3) if rec_cf is not None else None,
        "CBF": round(rec_cbf, 3) if rec_cbf is not None else None,
        "Hybrid": round(rec_hybrid, 3) if rec_hybrid is not None else None
    })

# creating and displaying dataframe
df_results = pd.DataFrame(results)
print("\n===== Recall@5 Table =====")
print(df_results)

# average per method
print("\n===== Average values Recall@5 =====")
print(df_results[['CF', 'CBF', 'Hybrid']].mean())
